# LeetCode 1243 — Array Transformation bằng Python

**Link đề:** https://leetcode.com/problems/array-transformation/
**Độ khó:** Easy · **Chủ đề:** Array, Simulation

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def transformArray(self, arr: List[int]) -> List[int]:
        ...
```

## Mục tiêu học
1. Làm quen với dạng bài **mô phỏng (simulation)** "theo ngày": trạng thái ngày mới tính **hoàn toàn từ ngày cũ**.
2. Hiểu bẫy kinh điển **cập nhật tại chỗ (in-place)** làm hỏng dữ liệu của ngày cũ, và 3 cách tránh: **mảng mới**, **copy**, **biến giữ giá trị cũ**.
3. Nắm `list.copy()`, `arr[:]`, so sánh list `==`, vòng `while True` + cờ `changed`, `zip` ba dãy.
4. Lý giải vì sao quá trình **chắc chắn dừng** (đại lượng "tổng dao động" giảm mỗi ngày) và ước lượng số ngày tối đa.
5. Trace từng ngày, test đối chiếu, bài tập TODO + đáp án cuối.

## 1. Đề bài và ví dụ

Cho mảng ban đầu `arr`. Mỗi ngày, ta tạo một mảng mới **dựa trên mảng của ngày hôm trước**. Vào ngày thứ `i`, với mảng của ngày `i - 1`:

1. Nếu một phần tử **nhỏ hơn cả** hàng xóm trái **và** hàng xóm phải → phần tử đó **tăng 1**.
2. Nếu một phần tử **lớn hơn cả** hàng xóm trái **và** hàng xóm phải → phần tử đó **giảm 1**.
3. Phần tử **đầu tiên và cuối cùng không bao giờ thay đổi**.

Sau một số ngày, mảng không thay đổi nữa. Trả về **mảng cuối cùng** đó.

| `arr` | Kết quả | Diễn biến |
|---|---|---|
| `[6, 2, 3, 4]` | `[6, 3, 3, 4]` | ngày 1: `2` nhỏ hơn 6 và 3 → tăng thành 3 |
| `[1, 6, 3, 4, 3, 5]` | `[1, 4, 4, 4, 4, 5]` | ngày 1: `[1, 5, 4, 3, 4, 5]`; ngày 2: `[1, 4, 4, 4, 4, 5]` |

**Constraints (theo hiểu biết):** `3 <= arr.length <= 100`; `1 <= arr[i] <= 100`.

Lưu ý chữ "**cả hai**" và "**nhỏ hơn / lớn hơn**" (nghiêm ngặt): phần tử bằng một hàng xóm thì không đổi.

## 2. Nền tảng Python cần biết

### 2.1. Truy cập hàng xóm bằng chỉ số

Phần tử `arr[i]` có hàng xóm `arr[i - 1]` và `arr[i + 1]`. Chỉ xét `i` từ `1` tới `n - 2` (bỏ đầu và cuối) → `range(1, n - 1)`.

In [1]:
arr = [1, 6, 3, 4, 3, 5]
n = len(arr)
for i in range(1, n - 1):
    left, mid, right = arr[i - 1], arr[i], arr[i + 1]
    kind = "đáy (min cục bộ)" if mid < left and mid < right else "đỉnh (max cục bộ)" if mid > left and mid > right else "-"
    print(f"i={i}: ({left}, {mid}, {right}) -> {kind}")

i=1: (1, 6, 3) -> đỉnh (max cục bộ)
i=2: (6, 3, 4) -> đáy (min cục bộ)
i=3: (3, 4, 3) -> đỉnh (max cục bộ)
i=4: (4, 3, 5) -> đáy (min cục bộ)


Một phần tử nhỏ hơn cả hai hàng xóm gọi là **cực tiểu cục bộ (local minimum)**, lớn hơn cả hai gọi là **cực đại cục bộ (local maximum)**.

Python cho phép **so sánh chuỗi**: `left > mid < right` nghĩa là `left > mid and mid < right`.

In [2]:
left, mid, right = 6, 2, 3
print("left > mid < right :", left > mid < right)
print("left < mid > right :", left < mid > right)

left > mid < right : True
left < mid > right : False


### 2.2. Gán list KHÔNG tạo bản sao — `copy()` và `[:]`

`b = a` chỉ tạo **tên mới** cho cùng một list. Sửa `b` là sửa luôn `a`. Muốn bản sao độc lập: `a.copy()`, `a[:]` hoặc `list(a)`.

In [3]:
a = [1, 2, 3]
alias = a
alias[0] = 99
print("a sau khi sửa alias:", a, "| alias is a:", alias is a)

a = [1, 2, 3]
c1, c2, c3 = a.copy(), a[:], list(a)
c1[0] = 99
print("a sau khi sửa bản copy:", a, "| c1:", c1, "| c1 is a:", c1 is a)

a sau khi sửa alias: [99, 2, 3] | alias is a: True
a sau khi sửa bản copy: [1, 2, 3] | c1: [99, 2, 3] | c1 is a: False


### 2.3. So sánh hai list bằng `==`

`==` so sánh **nội dung** từng phần tử; `is` so sánh **có phải cùng một đối tượng** không.

In [4]:
print([1, 2, 3] == [1, 2, 3], [1, 2, 3] is [1, 2, 3], [1, 2] == [2, 1])

True False False


### 2.4. `while True` + `break` và cờ `changed`

Mẫu "lặp đến khi ổn định": lặp mãi, mỗi vòng ghi nhận có thay đổi không; không đổi thì `break`.

In [5]:
x = 37
day = 0
while True:
    new_x = x // 2 if x > 5 else x
    if new_x == x:
        break
    x = new_x
    day += 1
    print("ngày", day, "-> x =", x)
print("ổn định ở", x)

ngày 1 -> x = 18
ngày 2 -> x = 9
ngày 3 -> x = 4
ổn định ở 4


## 3. Cách 1 — Mô phỏng thẳng: mỗi ngày tạo một mảng mới

Đọc đúng đề: ngày mới tính từ **mảng ngày cũ**. Cách an toàn nhất: mỗi ngày tạo `new = arr.copy()`, **đọc từ `arr`**, **ghi vào `new`**. Nếu `new == arr` → dừng.

In [6]:
def transform_simulate(arr: list[int]) -> list[int]:
    arr = arr.copy()                      # không sửa input của người gọi
    while True:
        new = arr.copy()
        for i in range(1, len(arr) - 1):
            if arr[i] < arr[i - 1] and arr[i] < arr[i + 1]:
                new[i] += 1
            elif arr[i] > arr[i - 1] and arr[i] > arr[i + 1]:
                new[i] -= 1
        if new == arr:                    # không còn thay đổi
            return arr
        arr = new

print(transform_simulate([6, 2, 3, 4]))
print(transform_simulate([1, 6, 3, 4, 3, 5]))

[6, 3, 3, 4]
[1, 4, 4, 4, 4, 5]


Chi phí: mỗi ngày `O(n)` (duyệt + copy + so sánh). Số ngày là bao nhiêu? Ta sẽ chứng minh ở mục 8 rằng nó bị chặn (≤ khoảng `n · max(arr) / 2`). Với `n, arr[i] ≤ 100` thì rất nhỏ.

## 4. Nút thắt và insight

### Nút thắt 1 — Cập nhật tại chỗ là **sai**

Nếu sửa `arr[i]` ngay trong vòng lặp, khi xét `arr[i + 1]` ta sẽ so với **giá trị mới** của `arr[i]` (của ngày hôm nay), trái đề bài. Ví dụ `[5, 1, 5, 1, 5]`:

In [7]:
def one_day_inplace_wrong(arr):
    arr = arr.copy()
    for i in range(1, len(arr) - 1):
        if arr[i] < arr[i - 1] and arr[i] < arr[i + 1]:
            arr[i] += 1
        elif arr[i] > arr[i - 1] and arr[i] > arr[i + 1]:
            arr[i] -= 1
    return arr

def one_day_correct(arr):
    new = arr.copy()
    for i in range(1, len(arr) - 1):
        if arr[i] < arr[i - 1] and arr[i] < arr[i + 1]:
            new[i] += 1
        elif arr[i] > arr[i - 1] and arr[i] > arr[i + 1]:
            new[i] -= 1
    return new

a = [5, 1, 5, 1, 5]
print("đúng (ngày 1):", one_day_correct(a))
print("sai  (ngày 1):", one_day_inplace_wrong(a))

đúng (ngày 1): [5, 2, 4, 2, 5]
sai  (ngày 1): [5, 2, 4, 2, 5]


Đúng: cả `1` (i=1, i=3) đều tăng, `5` ở giữa (i=2) là đỉnh → giảm: `[5, 2, 4, 2, 5]`.
Sai: sau khi `arr[1]` thành `2`, xét `i=2`: `5 > 2 và 5 > 1` vẫn giảm → 4; rồi `i=3`: `1 < 4 và 1 < 5` → 2. Ở ví dụ này tình cờ giống nhau! Hãy xem ví dụ lộ lỗi:

In [8]:
for a in ([4, 2, 3, 1, 5], [5, 4, 5, 3, 5, 5]):
    print(a, "-> đúng:", one_day_correct(a), "| sai:", one_day_inplace_wrong(a))
a = [4, 2, 3, 1, 5]
assert one_day_correct(a) != one_day_inplace_wrong(a)

[4, 2, 3, 1, 5] -> đúng: [4, 3, 2, 2, 5] | sai: [4, 3, 3, 2, 5]
[5, 4, 5, 3, 5, 5] -> đúng: [5, 5, 4, 4, 5, 5] | sai: [5, 5, 5, 4, 5, 5]


Với `[4, 2, 3, 1, 5]`: đúng là `2` tăng (đáy), `3` giảm (đỉnh), `1` tăng (đáy) → `[4, 3, 2, 2, 5]`. Bản sai: sau khi `2 → 3`, phần tử `3` ở i=2 so với hàng xóm trái **mới** là `3` → không còn là đỉnh → không giảm. Kết quả lệch.

### Insight — chỉ cần nhớ **giá trị cũ của hàng xóm trái**

Khi duyệt trái → phải, hàng xóm phải `arr[i + 1]` **chưa bị sửa**, chỉ hàng xóm trái `arr[i - 1]` có thể đã bị sửa. Vậy chỉ cần một biến `prev` lưu giá trị **cũ** của `arr[i - 1]` → cập nhật tại chỗ với bộ nhớ `O(1)` thêm.

## 5. Lời giải chính — giảng từng đoạn rồi ghép

### Cách 2 — Cập nhật tại chỗ với biến `prev`

**Đoạn 1:** `prev = arr[0]` (giá trị cũ của hàng xóm trái).

**Đoạn 2:** với mỗi `i`, lưu `cur = arr[i]` (giá trị cũ của chính nó) **trước khi** sửa.

**Đoạn 3:** so `cur` với `prev` và `arr[i + 1]` (vẫn là giá trị cũ) → tăng/giảm `arr[i]`, bật cờ `changed`.

**Đoạn 4:** `prev = cur` để bước sau dùng giá trị cũ.

**Đoạn 5:** lặp các ngày tới khi `changed` là `False`.

In [9]:
def transform_prev(arr: list[int]) -> list[int]:
    arr = arr.copy()
    changed = True
    while changed:                                   # đoạn 5
        changed = False
        prev = arr[0]                                # đoạn 1
        for i in range(1, len(arr) - 1):
            cur = arr[i]                             # đoạn 2
            if cur < prev and cur < arr[i + 1]:      # đoạn 3
                arr[i] += 1
                changed = True
            elif cur > prev and cur > arr[i + 1]:
                arr[i] -= 1
                changed = True
            prev = cur                               # đoạn 4
    return arr

print(transform_prev([6, 2, 3, 4]))
print(transform_prev([1, 6, 3, 4, 3, 5]))
print(transform_prev([4, 2, 3, 1, 5]))

[6, 3, 3, 4]
[1, 4, 4, 4, 4, 5]
[4, 3, 2, 2, 5]


### Cách 3 — Mảng mới bằng list comprehension + `zip` ba dãy

`zip(arr, arr[1:], arr[2:])` sinh các bộ ba `(trái, giữa, phải)` liên tiếp. Ta tính giá trị mới cho từng phần tử giữa, rồi ghép lại với phần tử đầu và cuối.

In [10]:
print(list(zip([1, 6, 3, 4], [1, 6, 3, 4][1:], [1, 6, 3, 4][2:])))

def next_day(arr: list[int]) -> list[int]:
    middle = [m + (l > m < r) - (l < m > r) for l, m, r in zip(arr, arr[1:], arr[2:])]
    return [arr[0]] + middle + [arr[-1]]

def transform_zip(arr: list[int]) -> list[int]:
    while True:
        new = next_day(arr)
        if new == arr:
            return arr
        arr = new

print(transform_zip([6, 2, 3, 4]), transform_zip([1, 6, 3, 4, 3, 5]))

[(1, 6, 3), (6, 3, 4)]
[6, 3, 3, 4] [1, 4, 4, 4, 4, 5]


Mẹo `m + (l > m < r) - (l < m > r)`: biểu thức so sánh trả `True`/`False`, cộng vào số thì thành `1`/`0`. Đáy → `+1`, đỉnh → `-1`, còn lại `+0`. Gọn nhưng hơi "khéo" — khi mới học, nên dùng `if/elif` cho rõ.

### Cách 4 — Đệ quy (minh hoạ tư duy "điểm bất động")

Mảng cuối là **điểm bất động** của phép biến đổi `next_day`: `next_day(x) == x`. Đệ quy: nếu chưa bất động thì gọi lại với ngày mới.

In [11]:
def transform_recursive(arr: list[int]) -> list[int]:
    new = next_day(arr)
    return arr if new == arr else transform_recursive(new)

print(transform_recursive([1, 6, 3, 4, 3, 5]))

[1, 4, 4, 4, 4, 5]


Với constraints nhỏ, độ sâu đệ quy (= số ngày) nhỏ hơn nhiều giới hạn mặc định ~1000 của Python. Nhưng với dữ liệu lớn hơn, nên dùng vòng lặp.

## 6. TRACE từng ngày

Ví dụ 2: `arr = [1, 6, 3, 4, 3, 5]`

**Ngày 1** (đọc từ `[1, 6, 3, 4, 3, 5]`):

| i | (trái, giữa, phải) | Loại | Giá trị mới |
|:---:|:---:|---|:---:|
| 1 | (1, 6, 3) | đỉnh | 5 |
| 2 | (6, 3, 4) | đáy | 4 |
| 3 | (3, 4, 3) | đỉnh | 3 |
| 4 | (4, 3, 5) | đáy | 4 |

→ `[1, 5, 4, 3, 4, 5]`

**Ngày 2** (đọc từ `[1, 5, 4, 3, 4, 5]`): i=1 `(1,5,4)` đỉnh → 4; i=2 `(5,4,3)` không; i=3 `(4,3,4)` đáy → 4; i=4 `(3,4,5)` không → `[1, 4, 4, 4, 4, 5]`.

**Ngày 3:** không phần tử nào là đỉnh/đáy nghiêm ngặt → dừng.

In [12]:
def trace_transform(arr: list[int]) -> list[int]:
    arr = arr.copy()
    day = 0
    print(f"ngày {day}: {arr}")
    while True:
        new = arr.copy()
        notes = []
        for i in range(1, len(arr) - 1):
            l, m, r = arr[i - 1], arr[i], arr[i + 1]
            if m < l and m < r:
                new[i] += 1
                notes.append(f"i={i} ({l},{m},{r}) đáy +1")
            elif m > l and m > r:
                new[i] -= 1
                notes.append(f"i={i} ({l},{m},{r}) đỉnh -1")
        if new == arr:
            print(f"ngày {day + 1}: không đổi -> dừng. Kết quả {arr}")
            return arr
        day += 1
        arr = new
        print(f"ngày {day}: {arr}   | " + "; ".join(notes))

assert trace_transform([1, 6, 3, 4, 3, 5]) == [1, 4, 4, 4, 4, 5]

ngày 0: [1, 6, 3, 4, 3, 5]
ngày 1: [1, 5, 4, 3, 4, 5]   | i=1 (1,6,3) đỉnh -1; i=2 (6,3,4) đáy +1; i=3 (3,4,3) đỉnh -1; i=4 (4,3,5) đáy +1
ngày 2: [1, 4, 4, 4, 4, 5]   | i=1 (1,5,4) đỉnh -1; i=3 (4,3,4) đáy +1
ngày 3: không đổi -> dừng. Kết quả [1, 4, 4, 4, 4, 5]


In [13]:
assert trace_transform([6, 2, 3, 4]) == [6, 3, 3, 4]
print()
assert trace_transform([2, 1, 2, 1, 1, 2, 2, 1]) == [2, 2, 1, 1, 1, 2, 2, 1]

ngày 0: [6, 2, 3, 4]
ngày 1: [6, 3, 3, 4]   | i=1 (6,2,3) đáy +1
ngày 2: không đổi -> dừng. Kết quả [6, 3, 3, 4]

ngày 0: [2, 1, 2, 1, 1, 2, 2, 1]
ngày 1: [2, 2, 1, 1, 1, 2, 2, 1]   | i=1 (2,1,2) đáy +1; i=2 (1,2,1) đỉnh -1
ngày 2: không đổi -> dừng. Kết quả [2, 2, 1, 1, 1, 2, 2, 1]


Ví dụ cuối cho thấy một trạng thái ổn định có thể vẫn "lồi lõm" (có đoạn bằng nhau): vì quy tắc dùng so sánh **nghiêm ngặt**, phần tử bằng hàng xóm thì đứng yên.

Một ví dụ cần **nhiều ngày** (đỉnh cao bị "mài" dần):

In [14]:
final = trace_transform([1, 9, 1])
assert final == [1, 1, 1]

ngày 0: [1, 9, 1]
ngày 1: [1, 8, 1]   | i=1 (1,9,1) đỉnh -1
ngày 2: [1, 7, 1]   | i=1 (1,8,1) đỉnh -1
ngày 3: [1, 6, 1]   | i=1 (1,7,1) đỉnh -1
ngày 4: [1, 5, 1]   | i=1 (1,6,1) đỉnh -1
ngày 5: [1, 4, 1]   | i=1 (1,5,1) đỉnh -1
ngày 6: [1, 3, 1]   | i=1 (1,4,1) đỉnh -1
ngày 7: [1, 2, 1]   | i=1 (1,3,1) đỉnh -1
ngày 8: [1, 1, 1]   | i=1 (1,2,1) đỉnh -1
ngày 9: không đổi -> dừng. Kết quả [1, 1, 1]


## 7. Code nộp LeetCode

### 7.1. Bản chính — cập nhật tại chỗ với `prev` (O(1) bộ nhớ thêm)

In [15]:
from typing import List

class Solution:
    def transformArray(self, arr: List[int]) -> List[int]:
        changed = True
        while changed:
            changed = False
            prev = arr[0]
            for i in range(1, len(arr) - 1):
                cur = arr[i]
                if cur < prev and cur < arr[i + 1]:
                    arr[i] += 1
                    changed = True
                elif cur > prev and cur > arr[i + 1]:
                    arr[i] -= 1
                    changed = True
                prev = cur
        return arr

sol = Solution()
print(sol.transformArray([6, 2, 3, 4]), sol.transformArray([1, 6, 3, 4, 3, 5]))

[6, 3, 3, 4] [1, 4, 4, 4, 4, 5]


(Bản nộp sửa thẳng `arr` — LeetCode chấp nhận; nếu muốn giữ input, thêm `arr = arr[:]` ở đầu.)

### 7.2. Bản mảng mới mỗi ngày (dễ đọc nhất)

In [16]:
class SolutionCopy:
    def transformArray(self, arr: List[int]) -> List[int]:
        while True:
            new = arr[:]
            for i in range(1, len(arr) - 1):
                if arr[i - 1] > arr[i] < arr[i + 1]:
                    new[i] += 1
                elif arr[i - 1] < arr[i] > arr[i + 1]:
                    new[i] -= 1
            if new == arr:
                return arr
            arr = new

assert SolutionCopy().transformArray([1, 6, 3, 4, 3, 5]) == [1, 4, 4, 4, 4, 5]
print("SolutionCopy OK")

SolutionCopy OK


### 7.3. Bản `zip`

In [17]:
class SolutionZip:
    def transformArray(self, arr: List[int]) -> List[int]:
        while True:
            new = [arr[0]] + [m + (l > m < r) - (l < m > r)
                              for l, m, r in zip(arr, arr[1:], arr[2:])] + [arr[-1]]
            if new == arr:
                return arr
            arr = new

assert SolutionZip().transformArray([6, 2, 3, 4]) == [6, 3, 3, 4]
print("SolutionZip OK")

SolutionZip OK


## 8. Vì sao đúng và vì sao chắc chắn dừng?

**Đúng (cách `prev`):** bất biến trong một ngày — khi xét chỉ số `i`, `prev` = giá trị **của ngày cũ** tại `i - 1` (vì ta lưu `cur` trước khi sửa và gán `prev = cur`), còn `arr[i + 1]` chưa bị đụng tới nên cũng là giá trị cũ. Vậy mọi quyết định đều dựa trên mảng ngày cũ, đúng như đề — tương đương cách tạo mảng mới.

**Kết quả cuối đúng định nghĩa:** vòng lặp chỉ dừng khi một ngày không có thay đổi ⇒ mảng là điểm bất động.

**Chắc chắn dừng:** xét đại lượng *tổng dao động* `V = Σ |arr[i+1] − arr[i]|`.
- Một đáy nghiêm ngặt tăng 1 (hoặc đỉnh giảm 1) làm **cả hai** khoảng cách tới hàng xóm giảm 1 nếu hàng xóm đứng yên.
- Hai phần tử kề nhau không thể cùng là đáy (hay cùng là đỉnh); nếu một đáy kề một đỉnh cùng thay đổi, chúng tiến lại gần nhau, khoảng cách giữa chúng không tăng.
- Kết quả: mỗi ngày có thay đổi, `V` giảm **ít nhất 2** (ta kiểm chứng thực nghiệm bên dưới). `V ≥ 0` và ban đầu `V ≤ (n − 1)·(max − min)` ⇒ số ngày ≤ `(n − 1)·(max − min) / 2` ≈ `100 · 99 / 2` < 5000.

Thêm nữa, mọi giá trị luôn nằm trong `[min(arr), max(arr)]` ban đầu: đáy chỉ tăng tới tối đa giá trị hàng xóm nhỏ hơn, đỉnh chỉ giảm tới tối thiểu hàng xóm lớn hơn.

In [18]:
import random

def total_variation(a):
    return sum(abs(a[i + 1] - a[i]) for i in range(len(a) - 1))

rng = random.Random(1243)
min_drop = None
max_days = 0
for _ in range(3000):
    a = [rng.randint(1, 100) for _ in range(rng.randint(3, 12))]
    lo, hi = min(a), max(a)
    days = 0
    while True:
        b2 = one_day_correct(a)
        if b2 == a:
            break
        drop = total_variation(a) - total_variation(b2)
        min_drop = drop if min_drop is None else min(min_drop, drop)
        assert lo <= min(b2) and max(b2) <= hi          # giá trị không vượt khoảng ban đầu
        a = b2
        days += 1
    max_days = max(max_days, days)
print("V giảm ít nhất mỗi ngày:", min_drop, "| số ngày lớn nhất quan sát:", max_days)
assert min_drop >= 2

V giảm ít nhất mỗi ngày: 2 | số ngày lớn nhất quan sát: 91


## 9. Độ phức tạp và so sánh

Gọi `n = len(arr)`, `D` = số ngày cho tới khi ổn định (`D ≤ (n − 1)·(max − min)/2`).

| Cách | Time | Space thêm | Ghi chú |
|---|---|---|---|
| Mảng mới mỗi ngày | `O(D · n)` | `O(n)` | dễ đúng nhất |
| **Tại chỗ + `prev`** | `O(D · n)` | `O(1)` | tiết kiệm bộ nhớ, cần cẩn thận |
| `zip` comprehension | `O(D · n)` | `O(n)` | gọn |
| Đệ quy | `O(D · n)` | `O(D · n)` (stack + mảng) | minh hoạ; không nên dùng khi `D` lớn |

Với `n ≤ 100`, giá trị ≤ 100: `D · n` cỡ vài trăm nghìn phép tính trong trường hợp xấu — rất nhanh.

## 10. Bộ test

Mẫu đề, độ dài tối thiểu 3, mảng đã ổn định, mảng hằng, răng cưa (zig-zag), đỉnh cao cần nhiều ngày, giá trị biên; đối chiếu mọi cách trên dữ liệu ngẫu nhiên và kiểm tra input không bị sửa (với các hàm demo).

In [19]:
impls = {
    "simulate": transform_simulate,
    "prev": transform_prev,
    "zip": transform_zip,
    "recursive": transform_recursive,
    "Solution": lambda a: Solution().transformArray(a[:]),
    "SolCopy": lambda a: SolutionCopy().transformArray(a[:]),
    "SolZip": lambda a: SolutionZip().transformArray(a[:]),
}
cases = [
    ([6, 2, 3, 4], [6, 3, 3, 4]),
    ([1, 6, 3, 4, 3, 5], [1, 4, 4, 4, 4, 5]),
    ([1, 9, 1], [1, 1, 1]),                    # độ dài 3, cần nhiều ngày
    ([5, 1, 5], [5, 5, 5]),
    ([1, 2, 3, 4], [1, 2, 3, 4]),              # đã ổn định (đơn điệu)
    ([7, 7, 7, 7], [7, 7, 7, 7]),              # hằng
    ([5, 1, 5, 1, 5], [5, 3, 3, 3, 5]),        # răng cưa
    ([1, 100, 1, 100], [1, 50, 51, 100]),      # giá trị biên: đỉnh và đáy "gặp nhau" ở giữa
    ([4, 2, 3, 1, 5], [4, 3, 2, 2, 5]),
]
for arr, expected in cases:
    for name, f in impls.items():
        original = arr[:]
        got = f(arr)
        assert got == expected, (name, arr, got, expected)
        assert arr == original, ("input bị sửa", name)
    print(f"{str(arr):28} -> {expected} PASS")

[6, 2, 3, 4]                 -> [6, 3, 3, 4] PASS
[1, 6, 3, 4, 3, 5]           -> [1, 4, 4, 4, 4, 5] PASS
[1, 9, 1]                    -> [1, 1, 1] PASS
[5, 1, 5]                    -> [5, 5, 5] PASS
[1, 2, 3, 4]                 -> [1, 2, 3, 4] PASS
[7, 7, 7, 7]                 -> [7, 7, 7, 7] PASS
[5, 1, 5, 1, 5]              -> [5, 3, 3, 3, 5] PASS
[1, 100, 1, 100]             -> [1, 50, 51, 100] PASS
[4, 2, 3, 1, 5]              -> [4, 3, 2, 2, 5] PASS


In [20]:
rng = random.Random(51)
for _ in range(500):
    a = [rng.randint(1, 100) for _ in range(rng.randint(3, 15))]
    res = {name: f(a) for name, f in impls.items()}
    first = res["simulate"]
    assert all(r == first for r in res.values()), (a, res)
    assert next_day(first) == first                    # kết quả là điểm bất động
big = [rng.randint(1, 100) for _ in range(100)]
print("n=100 ->", Solution().transformArray(big[:])[:10], "...")
print("500 mảng ngẫu nhiên: mọi cách khớp nhau, kết quả là điểm bất động")

n=100 -> [23, 71, 71, 62, 62, 67, 86, 86, 50, 50] ...
500 mảng ngẫu nhiên: mọi cách khớp nhau, kết quả là điểm bất động


## 11. Lỗi người mới hay gặp

### Lỗi 1: Cập nhật tại chỗ mà không giữ giá trị cũ

Đã minh hoạ ở mục 4: kết quả **một ngày** sai. Có khi kết quả **cuối cùng** cũng sai:

In [21]:
def wrong_transform_inplace(arr):
    arr = arr.copy()
    changed = True
    while changed:
        changed = False
        for i in range(1, len(arr) - 1):
            if arr[i] < arr[i - 1] and arr[i] < arr[i + 1]:     # arr[i-1] có thể đã là giá trị MỚI
                arr[i] += 1; changed = True
            elif arr[i] > arr[i - 1] and arr[i] > arr[i + 1]:
                arr[i] -= 1; changed = True
    return arr

rng = random.Random(3)
for _ in range(2000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(3, 7))]
    if wrong_transform_inplace(a) != transform_simulate(a):
        print("phản ví dụ:", a, "| sai:", wrong_transform_inplace(a), "| đúng:", transform_simulate(a))
        break

phản ví dụ: [5, 4, 5, 3, 5, 5] | sai: [5, 5, 5, 5, 5, 5] | đúng: [5, 5, 4, 4, 5, 5]


### Lỗi 2: `new = arr` (không copy) rồi sửa `new`

`new` và `arr` là **cùng một list** → y hệt lỗi 1, và điều kiện dừng `new == arr` **luôn đúng** sau ngày đầu → dừng quá sớm.

In [22]:
def wrong_alias(arr):
    arr = arr.copy()
    while True:
        new = arr                                   # SAI: không copy
        for i in range(1, len(arr) - 1):
            if arr[i] < arr[i - 1] and arr[i] < arr[i + 1]:
                new[i] += 1
            elif arr[i] > arr[i - 1] and arr[i] > arr[i + 1]:
                new[i] -= 1
        if new == arr:                              # luôn True vì cùng một list
            return arr
        arr = new

print("sai :", wrong_alias([1, 9, 1]), "| đúng:", transform_simulate([1, 9, 1]))

sai : [1, 8, 1] | đúng: [1, 1, 1]


### Lỗi 3: Duyệt cả phần tử đầu/cuối

`range(len(arr))` → `arr[i - 1]` với `i = 0` là `arr[-1]` (phần tử **cuối**, Python không báo lỗi!) và `arr[i + 1]` với `i = n - 1` gây `IndexError`. Đề cũng cấm đổi đầu/cuối. Dùng `range(1, len(arr) - 1)`.

### Lỗi 4: Dùng `<=` / `>=` thay vì `<` / `>`

Phần tử bằng hàng xóm sẽ bị đổi → có thể **dao động mãi không dừng**. Ví dụ `[1, 1, 1]` với `<=`: `1 <= 1` → tăng thành 2; ngày sau `2 >= 1` → giảm về 1; lặp vô hạn. (Không chạy trực tiếp ở đây để tránh treo notebook — ta chỉ chạy giới hạn 6 ngày.)

In [23]:
def wrong_non_strict_days(arr, max_days=6):
    for day in range(1, max_days + 1):
        new = arr[:]
        for i in range(1, len(arr) - 1):
            if arr[i] <= arr[i - 1] and arr[i] <= arr[i + 1]:
                new[i] += 1
            elif arr[i] >= arr[i - 1] and arr[i] >= arr[i + 1]:
                new[i] -= 1
        arr = new
        print("ngày", day, arr)

wrong_non_strict_days([1, 1, 1])

ngày 1 [1, 2, 1]
ngày 2 [1, 1, 1]
ngày 3 [1, 2, 1]
ngày 4 [1, 1, 1]
ngày 5 [1, 2, 1]
ngày 6 [1, 1, 1]


## 12. Mẹo nhận dạng pattern

- Đề có cụm **"mỗi ngày / mỗi bước, trạng thái mới dựa trên trạng thái cũ"**, **"cho đến khi không đổi"** → mô phỏng với **hai bộ đệm** (cũ/mới) hoặc **biến giữ giá trị cũ**.
- Luôn tự hỏi: **cập nhật tại chỗ có làm hỏng dữ liệu mình còn cần đọc không?** Nếu có → copy, hoặc lưu tạm, hoặc mã hoá trạng thái (như Game of Life dùng giá trị 2, 3 để vừa nhớ cũ vừa ghi mới).
- Hỏi thêm: **có chắc dừng không?** Tìm một đại lượng giảm ngặt (potential function).

Bài cùng họ: 289 Game of Life (cập nhật đồng thời trên lưới), 957 Prison Cells After N Days (mô phỏng + phát hiện chu kỳ), 838 Push Dominoes, 1275/1260 Shift 2D Grid.

## 13. Biến thể

### 13.1. Chỉ mô phỏng đúng `k` ngày

In [24]:
def transform_k_days(arr: list[int], k: int) -> list[int]:
    for _ in range(k):
        new = next_day(arr)
        if new == arr:            # ổn định sớm -> dừng luôn
            break
        arr = new
    return arr

print(transform_k_days([1, 9, 1], 3), transform_k_days([1, 9, 1], 100))

[1, 6, 1] [1, 1, 1]


### 13.2. Đếm số ngày tới khi ổn định

In [25]:
def days_to_stable(arr: list[int]) -> int:
    days = 0
    while True:
        new = next_day(arr)
        if new == arr:
            return days
        arr, days = new, days + 1

print(days_to_stable([6, 2, 3, 4]), days_to_stable([1, 6, 3, 4, 3, 5]), days_to_stable([1, 9, 1]))

1 2 8


### 13.3. Prison Cells After N Days (LeetCode 957) — mô phỏng + phát hiện chu kỳ

Khi `N` rất lớn (`10^9`) không thể mô phỏng từng ngày; nhưng số trạng thái hữu hạn → sẽ lặp lại. Lưu `seen[state] = ngày` để phát hiện chu kỳ rồi nhảy cóc.

In [26]:
def prison_after_n_days(cells: list[int], n: int) -> list[int]:
    def step(c):
        return [0] + [int(c[i - 1] == c[i + 1]) for i in range(1, 7)] + [0]
    seen = {}
    day = 0
    while day < n:
        key = tuple(cells)
        if key in seen:
            cycle = day - seen[key]
            n = day + (n - day) % cycle       # bỏ qua các chu kỳ trọn vẹn
            seen = {}
            if day >= n:
                break
        seen[tuple(cells)] = day
        cells = step(cells)
        day += 1
    return cells

print(prison_after_n_days([0, 1, 0, 1, 1, 0, 0, 1], 7))
print(prison_after_n_days([1, 0, 0, 1, 0, 0, 1, 0], 1000000000))
assert prison_after_n_days([0, 1, 0, 1, 1, 0, 0, 1], 7) == [0, 0, 1, 1, 0, 0, 0, 0]
assert prison_after_n_days([1, 0, 0, 1, 0, 0, 1, 0], 1000000000) == [0, 0, 1, 1, 1, 1, 1, 0]

[0, 0, 1, 1, 0, 0, 0, 0]
[0, 0, 1, 1, 1, 1, 1, 0]


`tuple(cells)` — list không dùng làm key của dict được (vì có thể thay đổi), nên đổi sang tuple (bất biến).

## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `arr = [3, 7, 2, 8, 4]`. Viết mảng sau ngày 1, ngày 2, ... đến khi ổn định.

**Bài 2 (dự đoán output).** `a = [1, 2]; b = a; b.append(3); print(a)`; `[1, 2] is [1, 2]`; `3 > 1 < 2`.

**Bài 3 (điền TODO).** Hoàn thiện hàm tính **một ngày** (trả mảng mới, không sửa input).

In [27]:
def exercise_one_day(arr: list[int]) -> list[int]:
    new = arr.copy()
    for i in range(1, len(arr) - 1):
        # TODO 1: nếu arr[i] là đáy nghiêm ngặt -> new[i] += 1
        # TODO 2: nếu arr[i] là đỉnh nghiêm ngặt -> new[i] -= 1
        pass
    return new

# assert exercise_one_day([1, 6, 3, 4, 3, 5]) == [1, 5, 4, 3, 4, 5]

**Bài 4 (viết lại).** Viết lời giải đầy đủ dùng **cập nhật tại chỗ với `prev`**.

In [28]:
def exercise_transform_prev(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_transform_prev([1, 6, 3, 4, 3, 5]) == [1, 4, 4, 4, 4, 5]
# assert exercise_transform_prev([4, 2, 3, 1, 5]) == [4, 3, 2, 2, 5]

**Bài 5 (biến thể).** Mảng **vòng (circular)**: phần tử đầu có hàng xóm trái là phần tử cuối và ngược lại; **mọi** phần tử đều có thể đổi. Trả mảng ổn định.

In [29]:
def exercise_transform_circular(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_transform_circular([1, 5, 1, 5]) == [3, 3, 3, 3]
# assert exercise_transform_circular([2, 2, 2]) == [2, 2, 2]

**Bài 6 (mini challenge).** Trả về danh sách **tất cả trạng thái** từ ngày 0 tới ngày ổn định (bao gồm cả hai đầu), dùng để vẽ/animate quá trình.

In [30]:
def exercise_history(arr: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert exercise_history([6, 2, 3, 4]) == [[6, 2, 3, 4], [6, 3, 3, 4]]
# assert len(exercise_history([1, 9, 1])) == 9

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Xác định đỉnh/đáy theo mảng **cũ** | ghi mảng mới mỗi ngày | dừng khi một ngày không đổi |
| 2 | gán list là tạo bí danh | `is` khác `==` | so sánh chuỗi = `and` |
| 3 | Đọc `arr`, ghi `new` | `arr[i-1] > arr[i] < arr[i+1]` | `elif` cho đỉnh |
| 4 | Lưu giá trị cũ trước khi sửa | `prev`, `cur`, `changed` | `prev = cur` ở cuối mỗi bước |
| 5 | Hàng xóm vòng | `arr[(i - 1) % n]`, `arr[(i + 1) % n]` | duyệt `range(n)` với mảng mới mỗi ngày |
| 6 | Lưu lịch sử | `history = [arr]` | `while` thêm `new` cho tới khi `new == history[-1]` |

## 16. Đáp án tham khảo

**Bài 1:** ngày 0 `[3, 7, 2, 8, 4]` → ngày 1 `[3, 6, 3, 7, 4]` → ngày 2 `[3, 5, 4, 6, 4]` → ngày 3 `[3, 4, 5, 5, 4]` → ngày 4 không đổi (`5, 5` bằng nhau, `4` không phải đáy nghiêm ngặt). Kết quả `[3, 4, 5, 5, 4]`.

**Bài 2:** `[1, 2, 3]`; `False`; `True`.

In [31]:
def answer_one_day(arr):
    new = arr.copy()
    for i in range(1, len(arr) - 1):
        if arr[i - 1] > arr[i] < arr[i + 1]:
            new[i] += 1
        elif arr[i - 1] < arr[i] > arr[i + 1]:
            new[i] -= 1
    return new

def answer_transform_prev(arr):
    arr = arr.copy()
    changed = True
    while changed:
        changed = False
        prev = arr[0]
        for i in range(1, len(arr) - 1):
            cur = arr[i]
            if prev > cur < arr[i + 1]:
                arr[i] += 1; changed = True
            elif prev < cur > arr[i + 1]:
                arr[i] -= 1; changed = True
            prev = cur
    return arr

def answer_transform_circular(arr):
    n = len(arr)
    while True:
        new = arr.copy()
        for i in range(n):
            l, m, r = arr[(i - 1) % n], arr[i], arr[(i + 1) % n]
            if l > m < r:
                new[i] += 1
            elif l < m > r:
                new[i] -= 1
        if new == arr:
            return arr
        arr = new

def answer_history(arr):
    history = [arr.copy()]
    while True:
        new = answer_one_day(history[-1])
        if new == history[-1]:
            return history
        history.append(new)

assert transform_simulate([3, 7, 2, 8, 4]) == [3, 4, 5, 5, 4]
assert answer_history([3, 7, 2, 8, 4]) == [[3, 7, 2, 8, 4], [3, 6, 3, 7, 4], [3, 5, 4, 6, 4], [3, 4, 5, 5, 4]]
a_ = [1, 2]; b_ = a_; b_.append(3)
assert (a_, [1, 2] is [1, 2], 3 > 1 < 2) == ([1, 2, 3], False, True)
assert answer_one_day([1, 6, 3, 4, 3, 5]) == [1, 5, 4, 3, 4, 5]
assert answer_transform_prev([1, 6, 3, 4, 3, 5]) == [1, 4, 4, 4, 4, 5]
assert answer_transform_prev([4, 2, 3, 1, 5]) == [4, 3, 2, 2, 5]
assert answer_transform_circular([1, 5, 1, 5]) == [3, 3, 3, 3]
assert answer_transform_circular([2, 2, 2]) == [2, 2, 2]
assert answer_history([6, 2, 3, 4]) == [[6, 2, 3, 4], [6, 3, 3, 4]]
assert len(answer_history([1, 9, 1])) == 9
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
while True:
    new = arr[:]                              # đọc từ arr, ghi vào new
    for i in range(1, len(arr) - 1):
        if arr[i-1] > arr[i] < arr[i+1]:   new[i] += 1
        elif arr[i-1] < arr[i] > arr[i+1]: new[i] -= 1
    if new == arr:
        return arr
    arr = new
```

**Câu thần chú:** *"Ngày mới chỉ nhìn ngày cũ — đọc một bản, ghi một bản."*

### Checklist trước khi submit
- [ ] Không cập nhật tại chỗ mà đọc nhầm giá trị mới (dùng mảng mới hoặc `prev`)?
- [ ] `new = arr[:]` / `arr.copy()`, không phải `new = arr`?
- [ ] Bỏ qua phần tử đầu và cuối (`range(1, n - 1)`)?
- [ ] So sánh **nghiêm ngặt** `<` / `>`?
- [ ] Điều kiện dừng: một ngày không có thay đổi?
- [ ] Tên method `transformArray(self, arr)`?